# Novelty experiments on Kaggle / Colab

Runs one shard of the novelty experiments (see `extended_studies/novelty/README.md`).
Turn on a GPU (Kaggle: Settings → Accelerator → GPU T4). Set `SHARD` below, e.g. run four copies of this notebook with `0/4`, `1/4`, `2/4`, `3/4`.
When it finishes, download `results_<shard>.zip` from the Output panel and unzip all shards into `extended_studies/novelty/results/`.

In [ ]:
EXPS = "E1 E2a E2b E3a E3b E3c E3d"   # or a subset, e.g. "E1"
SIZE = "full"                        # "quick" for a pilot
SHARD = "0/4"
REPO = "https://github.com/bloodrepaer/IE643-Binary-Brains-PINN-Inversion.git" 

In [ ]:
import os, subprocess
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "-q", REPO, "repo"], check=True)
os.chdir("repo/extended_studies")
subprocess.run("pip -q install sympy tabulate", shell=True)
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# data (about 30 min on Kaggle CPU; each shard builds what its experiments need)
need = set(EXPS.split())
steps = ["core/generate_dataset_v2.py"]
if "E1" in need: steps.append("study3_pinn_favoured/generate_pinn_favoured.py")
if "E3b" in need: steps += ["study1_representation/extend_sensor_density.py", "novelty/sensor_design.py --size full"]
if "E2b" in need: steps.append("novelty/arrhenius.py --size full")
for s in steps:
    print(">>", s, flush=True)
    subprocess.run(f"python run.py {s}", shell=True, check=True)

In [ ]:
subprocess.run("python run.py novelty/test_novelty.py", shell=True, check=True)

In [ ]:
os.environ["NOVELTY_RESULTS"] = "/kaggle/working/results" if os.path.exists("/kaggle/working") else os.path.abspath("novelty/results")
subprocess.run(f"python run.py novelty/run_novelty.py --exp {EXPS} --size {SIZE} --shard {SHARD}", shell=True)

In [ ]:
tag = SHARD.replace("/", "of")
out = os.environ["NOVELTY_RESULTS"]
subprocess.run(f"cd {os.path.dirname(out)} && zip -qr results_{tag}.zip {os.path.basename(out)}", shell=True, check=True)
print("download", os.path.join(os.path.dirname(out), f"results_{tag}.zip"))